# 04 — Доверительные интервалы стоимости и практических границ

По умолчанию анализируется новый эксперимент из 03: `ANALYSIS_MODE="fresh"`.
Симуляция и обучение здесь не запускаются. Для прежних графиков исходного скана
выберите `ANALYSIS_MODE="legacy"`.

Интервалы μ описывают **сеточные границы в пределах рассчитанной сетки**, а не границы
истинного оптимального значения на всей оси. Отсутствие интервала отображается явно.


In [ ]:
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from osfbm.boundaries import boundaries_from_scan
from osfbm.core import NodeResult
from osfbm.results import load_run

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"
INK, MUTED = "#0b0b0b", "#8a8a86"

plt.rcParams.update({
    "figure.figsize": (7.5, 4.2), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "axes.titlesize": 11,
    "axes.grid": True, "grid.color": "#e8e8e4", "grid.linewidth": 0.8,
    "lines.linewidth": 1.8, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "font.size": 9,
    "legend.frameon": False,
})
from osfbm.experiment import load_experiment
from osfbm.confidence_plots import plot_audit, STATUS_LABELS
ANALYSIS_MODE = "legacy"


In [ ]:
RUN_ID = "fresh_train100k_test100k_v1"
H = 0.3
EPSILON = 0.03
ANALYSIS_MODE = "fresh"  # legacy — прежний формат результатов
RUN_DIR = ROOT / "result_optimal_stopping" / "runs" / RUN_ID


## Анализ независимого теста

Параметры доверия берутся из нового эксперимента; ε задаётся выше. Незавершённые узлы
можно просмотреть в таблице, но они не используются для финальных графиков и
интервалов μ. Повторный запуск анализа не меняет семейство проверяемых узлов.


In [ ]:
if ANALYSIS_MODE == "fresh":
    manifest, nodes, boundaries = load_experiment(RUN_DIR, epsilon=EPSILON)
    print("Train:", manifest["config"]["M_train"], "Test:", manifest["config"]["M_test"])
    print("Завершено узлов:", int(nodes.complete.sum()), "/", manifest["family_size"])
    display(nodes[nodes.H.eq(H)])
    report = boundaries.copy()
    report["status_description"] = report.status.map(STATUS_LABELS)
    display(report)
    nodes, boundaries, figures = plot_audit(RUN_DIR, H=H, epsilon=EPSILON)
    plt.show()
    print("Таблицы и графики:", RUN_DIR / "analysis" / f"epsilon_{EPSILON:g}")
elif ANALYSIS_MODE != "legacy":
    raise ValueError("ANALYSIS_MODE должен быть fresh или legacy")


## Как читать интервалы

На независимом тесте стратегия даёт награды $Y_i=\mu\tau_i+B^H_{\tau_i}$.
Вычисляем $\bar Y$, выборочное $s^2=\sum_i(Y_i-\bar Y)^2/(M-1)$ и $SE=s/\sqrt M$.
Точечный интервал: $\bar Y\pm t_{M-1,\,0.975}SE$ при доверии 95%.

Для $K$ заранее выбранных узлов общая полоса использует
$t_{M-1,\,1-0.05/(2K)}$. Она шире точечного интервала и учитывает проверку
всего семейства. Общие траектории по μ не мешают поправке Бонферрони.
Награды не обязаны быть нормальными: покрытие приближённое при большом M.

Для первой границы сравниваем полосу стоимости с ε, для второй — полосу
стоимости минус μT. Узел гарантированно внутри пороговой области, если верхний
конец полосы ≤ ε; возможно внутри, если нижний конец ≤ ε. Последние/первые
такие узлы дают пределы положения сеточной границы. Между узлами не интерполируем.

Если переход не локализован внутри сетки, потребуется расширить её диапазон.
Повторные переходы отмечаются как неоднозначность; монотонность не навязывается.
Совпавшие концы интервала означают определённый **узел сетки**, а не нулевую
ошибку непрерывной границы. Не учитываются обучение, дискретизация и ошибка
относительно истинного оптимума.

Подробнее: [формулы и примеры](../docs/05-confidence-intervals.md).


In [ ]:
if ANALYSIS_MODE == "legacy":
    # RUN_ID=None позволяет выбрать последний сохранённый запуск.
    if RUN_ID is None:
        runs = sorted((ROOT / "result_optimal_stopping" / "runs").glob("*/config.json"))
        if not runs:
            raise FileNotFoundError("Нет сохранённых запусков. Сначала выполните 03-grid.ipynb")
        RUN_ID = runs[-1].parent.name
    
    RUN_DIR = ROOT / "result_optimal_stopping" / "runs" / RUN_ID
    cfg, nodes = load_run(RUN_DIR)
    view = nodes[nodes["H"].eq(H)].sort_values("mu").copy()
    if view.empty:
        raise ValueError(f"Нет данных для H={H}. Доступные H: {sorted(nodes['H'].unique())}")
    print(f"Запуск: {RUN_ID}; H={H:g}; ε={EPSILON:g}; T={cfg.T}")
    print(f"Сохранено {len(view)} / {len(cfg.mu_grid)} узлов μ для выбранного H")


## Прежний анализ (`ANALYSIS_MODE="legacy"`)

Эти ячейки выполняются только в режиме legacy. Ниже сохранены прежние диагностики.

## Практические границы и значения по μ

Отрицательная разность границ означает перекрытие областей близости к двум
опорным стратегиям. Нелокализованные границы остаются пустыми.

In [ ]:
if ANALYSIS_MODE == "legacy":
    records = [NodeResult(**row) for row in view.to_dict("records")]
    boundary = boundaries_from_scan(records, cfg, epsilon=EPSILON)
    if len(view) < len(cfg.mu_grid):
        boundary = replace(boundary, note=(
            f"Неполная сетка: {len(view)}/{len(cfg.mu_grid)} узлов; " + boundary.note
        ))
    summary = pd.DataFrame([boundary.as_row()])
    display(summary)
    display(view[["mu", "V", "SE", "p0", "p1", "E_tau"]])
    
    OUT = RUN_DIR / "analysis" / f"H_{float(H)}" / f"epsilon_{float(EPSILON)}"
    OUT.mkdir(parents=True, exist_ok=True)
    summary.to_csv(OUT / "boundaries.csv", index=False)


## Стоимость и превышение опорных значений

Полоса ±SE показывает Monte Carlo ошибку оценки стоимости. Вертикальные
линии — найденные ε-границы, затенённые интервалы - разрешение сетки.

In [ ]:
if ANALYSIS_MODE == "legacy":
    mu = view["mu"].to_numpy()
    value = view["V"].to_numpy()
    se = view["SE"].to_numpy()
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(mu, value, color=C1, label="V̂")
    axes[0].fill_between(mu, value - se, value + se, color=C1, alpha=0.15, label="±SE")
    axes[0].plot(mu, np.maximum(0, mu * cfg.T), "--", color=MUTED, label="max(0, μT)")
    axes[0].set_title("Оценка стоимости")
    axes[0].set_ylabel("V̂")
    axes[1].plot(mu, value, color=C1, label="V̂ − 0")
    axes[2].plot(mu, value - mu * cfg.T, color=C1, label="V̂ − μT")
    for ax, title in zip(axes[1:], ["Превышение над выходом в нуле", "Превышение над удержанием"]):
        ax.axhline(EPSILON, color="black", linestyle="--", label=f"ε={EPSILON:g}")
        ax.axhline(0, color=MUTED, linewidth=0.8)
        ax.set_title(title)
        ax.set_ylabel("Превышение стоимости")
    for position, bracket, label, color, panel in [
        (boundary.mu1_epsilon, boundary.mu1_bracket, "μ₁ᵋ", C2, 1),
        (boundary.mu2_epsilon, boundary.mu2_bracket, "μ₂ᵋ", C3, 2),
    ]:
        if position is not None:
            for ax in (axes[0], axes[panel]):
                ax.axvline(position, color=color, linestyle=":", label=label)
                ax.axvspan(*bracket, color=color, alpha=0.12)
    for ax in axes:
        ax.set_xlabel("μ")
        ax.legend(fontsize=8)
    fig.suptitle(f"H={H:g}, ε={EPSILON:g}")
    fig.tight_layout()
    fig.savefig(OUT / "values.png", dpi=160, bbox_inches="tight")


## Диагностики остановки

Доли остановок, среднее время и численный наклон относятся к той же стратегии.
Близость стоимости к опорной не означает одинаковый момент остановки всех
траекторий; точного совпадения наклона со средним временем здесь не требуется.

In [ ]:
if ANALYSIS_MODE == "legacy":
    fig, ax = plt.subplots()
    ax.plot(mu, view["p0"], label="p0 — выход в нуле")
    ax.plot(mu, view["p1"], label="p1 — удержание до конца")
    ax.plot(mu, view["E_tau"] / cfg.T, label="Среднее время / T")
    if len(view) >= 2:
        ax.plot(mu, np.gradient(value, mu) / cfg.T, "--", label="Численный наклон V̂ / T")
    ax.set(xlabel="μ", ylabel="Доля / нормированный наклон", title=f"Диагностики H={H:g}, ε={EPSILON:g}")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "diagnostics.png", dpi=160, bbox_inches="tight")
    print(f"Анализ сохранён в {OUT}")


## Границы по всем H — boundaries.png

Этот блок использует все сохранённые H выбранного `RUN_ID` и один заданный
`EPSILON`. Параметр `H` из основного анализа здесь не ограничивает выборку.
Повторного обучения нет. Можно выполнить блок после ячейки загрузки данных.

Слева — практические ε-границы, справа — их разность. Отрицательная разность
означает перекрытие областей близости. Пропуски и нелокализованные границы
отображаются разрывами; подробности приведены в таблице.

Файлы сохраняются в `analysis/epsilon_<EPSILON>/` внутри каталога запуска.

In [ ]:
if ANALYSIS_MODE == "legacy":
    grid_rows = []
    for h in sorted(cfg.H_grid):
        group = nodes[nodes["H"].eq(h)].sort_values("mu")
        if group.empty:
            grid_rows.append({
                "H": h, "epsilon": EPSILON,
                "mu1_epsilon": np.nan, "mu2_epsilon": np.nan,
                "boundary_difference": np.nan,
                "note": "Нет сохранённых узлов для этого H",
            })
            continue
        grid_boundary = boundaries_from_scan(
            [NodeResult(**row) for row in group.to_dict("records")],
            cfg, epsilon=EPSILON,
        )
        if len(group) < len(cfg.mu_grid):
            grid_boundary = replace(grid_boundary, note=(
                f"Неполная сетка: {len(group)}/{len(cfg.mu_grid)} узлов; " + grid_boundary.note
            ))
        grid_rows.append(grid_boundary.as_row())
    
    grid_summary = pd.DataFrame(grid_rows)
    display(grid_summary)
    
    fig_boundaries, axes_boundaries = plt.subplots(1, 2, figsize=(11, 4))
    for column, label, color in [
        ("mu1_epsilon", "μ₁ᵋ", C1),
        ("mu2_epsilon", "μ₂ᵋ", C2),
    ]:
        values = pd.to_numeric(grid_summary[column], errors="coerce").to_numpy(dtype=float)
        axes_boundaries[0].plot(grid_summary["H"], values, marker="o", color=color, label=label)
    difference = pd.to_numeric(grid_summary["boundary_difference"], errors="coerce").to_numpy(dtype=float)
    axes_boundaries[1].plot(grid_summary["H"], difference, marker="D", color=C3, label="μ₂ᵋ − μ₁ᵋ")
    for ax in axes_boundaries:
        ax.axhline(0, color=MUTED, linewidth=0.8)
        ax.set_xlabel("H")
        ax.legend()
    axes_boundaries[0].set(ylabel="μ", title="Практические границы")
    axes_boundaries[1].set(ylabel="Разность границ", title="Отрицательное значение: перекрытие")
    fig_boundaries.suptitle(f"ε={EPSILON:g}, T={cfg.T:g}")
    fig_boundaries.tight_layout()
    
    GRID_OUT = RUN_DIR / "analysis" / f"epsilon_{float(EPSILON)}"
    GRID_OUT.mkdir(parents=True, exist_ok=True)
    grid_summary.to_csv(GRID_OUT / "boundaries.csv", index=False)
    fig_boundaries.savefig(GRID_OUT / "boundaries.png", dpi=200, bbox_inches="tight")
    print(f"График: {GRID_OUT / 'boundaries.png'}")
